In [4]:
#Ao rodar esta parte do código é criado uma pasta no Drive
!pip install -q googlemaps openpyxl

# Importar bibliotecas
import pandas as pd
import googlemaps
from google.colab import drive

# Montar o Google Drive
drive.mount('/content/drive')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Ao rodar este código o processmaneto é iniciado e salvo



# Função para converter decimal para GMS
def decimal_para_gms(valor, direcao_pos, direcao_neg):
    direcao = direcao_pos if valor >= 0 else direcao_neg
    valor = abs(valor)
    graus = int(valor)
    minutos_dec = (valor - graus) * 60
    minutos = int(minutos_dec)
    segundos = round((minutos_dec - minutos) * 60)
    return f"{graus}°{minutos}'{segundos}\"{direcao}"

# Sua chave da API do Google Maps
API_KEY = 'sua-chave'  # É NECESSARIO CRIAR UMA CHAVE API DO GOOGLE

# Inicializar o cliente Google Maps
gmaps = googlemaps.Client(key=API_KEY)

# Caminho do arquivo no Google Drive
caminho_arquivo = '/content/drive/MyDrive/bases_ICMBio_GRs.xlsx'  # <- ajuste se o nome do arquivo for diferente / OU PASTA
aba = 'EscritóriosICMBio'
df = pd.read_excel(caminho_arquivo, sheet_name=aba)

# Função para preencher coordenadas
def preencher_coordenadas(row):
    if pd.isna(row['Latitude']) or pd.isna(row['Longitude']):
        try:
            resultado = gmaps.geocode(row['ENDERECO'])
            if resultado:
                location = resultado[0]['geometry']['location']
                lat = location['lat']
                lon = location['lng']
                lat_gms = decimal_para_gms(lat, 'N', 'S')
                lon_gms = decimal_para_gms(lon, 'E', 'W')
                return pd.Series([lat_gms, lon_gms])
        except Exception as e:
            print(f"Erro ao geocodificar '{row['ENDERECO']}': {e}")
    return pd.Series([row['Latitude'], row['Longitude']])

# Aplicar a função para preencher coordenadas
df[['Latitude', 'Longitude']] = df.apply(preencher_coordenadas, axis=1)

# Caminho para salvar no Google Drive
caminho_saida = '/content/drive/MyDrive/planilha_completada_gms.xlsx'
df.to_excel(caminho_saida, index=False)

print(f"Arquivo salvo em: {caminho_saida}")

/usr/local/lib/python3.11/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Web Extension extension is not supported and will be removed
  warn(msg)


Arquivo salvo em: /content/drive/MyDrive/planilha_completada_gms.xlsx
